# Clase 20 presencial y final: laboratorio integrador

## Pregunta inicial

**Ante un problema nuevo, ¿cómo identificamos la operación dominante y elegimos la estructura de datos y el algoritmo adecuados?**

<div style="border:1px solid #9bb8d3;border-left:6px solid #315f8c;background:#eef5fb;color:#17212b;padding:14px 18px;margin:14px 0;border-radius:4px;line-height:1.55"><strong style="color:#0f2f4f">INSTRUCCIÓN — Laboratorio final</strong><br>Predice antes de revelar, argumenta cada descarte y responde en <code style="color:#102a43;background:#dbeaf5;padding:2px 5px;border-radius:3px;font-weight:600">notebook.md</code>. No respondas dentro de este archivo.</div>

**Responde esta pregunta en notebook.md.**


## Objetivos de aprendizaje

Al terminar podrás:

- distinguir camino mínimo, conexión mínima global y orden de dependencias;
- determinar dirección, dominio de pesos y significado de aristas;
- elegir entre BFS, 0-1 BFS, Dijkstra, Kruskal y Kahn;
- relacionar cola, deque, heap, Union-Find y grados de entrada con su operación dominante;
- verificar contratos, reconocer casos fuera del alcance y rechazar alternativas;
- reutilizar implementaciones anteriores mediante adaptadores estrechos;
- diseñar pruebas que distinguen decisiones aparentemente similares;
- comunicar complejidad, interpretación y límites con precisión.


## 1. Recuperación: primero aparecen las estructuras

En la pantalla solo aparecen cinco recursos: **cola**, **deque**, **heap**, **Union-Find** y **grados de entrada**. No son nombres para memorizar de manera aislada: cada uno abarata una operación repetida.

| Recurso | Operación que vuelve barata |
| --- | --- |
| Cola | retirar en orden de llegada |
| Deque | agregar y retirar por ambos extremos |
| Heap | consultar y extraer una prioridad extrema |
| Union-Find | preguntar si dos nodos ya están conectados y fusionarlos |
| Grados de entrada + cola | detectar tareas sin requisitos pendientes |

**Actividad diagnóstica.** Sin nombrar algoritmos todavía, escribe junto a cada recurso una situación de las Clases 16–19 donde fue necesario. Después compara: dos algoritmos pueden compartir cola y aun así mantener invariantes distintos.

> [!IMPORTANT]
> La estructura no se elige por familiaridad, sino por la operación que domina el costo.

El siguiente paso es convertir esa intuición en un método que pueda repetirse ante un problema nuevo.

### Comprueba tu comprensión

**Pregunta:** ¿Qué operación repetida justifica cada una de las cinco estructuras mostradas al inicio?

**Responde esta pregunta en notebook.md.**


## 2. Del enunciado a una decisión

Un enunciado mezcla historia, datos, restricciones y objetivo. Antes de programar, aplica siempre esta ruta:

```text
problema → objetivo → tipo de grafo → restricciones → operación dominante
         → estructura → algoritmo → contrato → prueba → complejidad → interpretación
```

Ejemplo: “encontrar la ruta más rápida entre dos estaciones con tiempos no negativos”. La palabra *ruta* sugiere un camino; “más rápida” indica minimizar suma de pesos; “no negativos” habilita una propiedad específica. La operación repetida será extraer la menor distancia tentativa, por lo que un heap y Dijkstra forman una pareja coherente.

> [!WARNING]
> Saltar desde una palabra del enunciado hasta un algoritmo produce asociaciones frágiles. “Conectar” puede significar alcanzabilidad, una ruta o conectar toda una red.

Ahora separaremos el primer componente de la ruta: qué salida pide realmente el problema.

### Comprueba tu comprensión

**Pregunta:** ¿Por qué es peligroso elegir un algoritmo únicamente por una palabra clave del enunciado?

**Responde esta pregunta en notebook.md.**


## 3. Identificar el objetivo

Tres objetivos dominaron el cierre del curso:

- **camino mínimo:** optimizar una ruta entre un origen y otros nodos;
- **conexión mínima global:** conectar todos los nodos minimizando el costo total de la red;
- **orden de dependencias:** producir una secuencia que respete precedencias.

Considera una red de cuatro sedes. “Llegar barato de A a D” fija origen y destino; “tender la red total más barata” no privilegia un origen; “instalar equipos después de sus requisitos” ni siquiera minimiza una suma. Aunque los tres enunciados usan grafos, sus salidas no son intercambiables.

**Actividad.** Para cada frase, escribe la forma de la salida: una ruta, un conjunto de aristas o una secuencia de nodos. Esa forma suele revelar el objetivo antes de pensar en código.

Ya sabemos qué resultado buscar; ahora debemos interpretar si una arista puede recorrerse en ambos sentidos.

### Comprueba tu comprensión

**Pregunta:** ¿Qué diferencia observable hay entre la salida de un camino mínimo, un MST y un orden topológico?

**Responde esta pregunta en notebook.md.**


## 4. Dirección y significado de las aristas

Una arista no solo une nodos: expresa una relación. En una calle de un solo sentido, `A → B` no autoriza `B → A`. En un cable entre sedes, la conexión suele ser simétrica. En un prerrequisito, invertir la flecha cambia quién bloquea a quién.

```text
Ruta dirigida:       A ──→ B
Conexión no dirigida: A ─── B
Dependencia:     estudiar ──→ presentar
```

Kruskal, tal como se estudió, requiere un grafo no dirigido. Kahn requiere precedencias dirigidas. BFS, 0-1 BFS y Dijkstra pueden operar en ambos casos si la representación contiene las aristas permitidas.

> [!CAUTION]
> Duplicar automáticamente cada arista para “hacerlo más fácil” destruye el significado de calles y dependencias dirigidas.

Con la dirección establecida, falta clasificar los pesos sin perder información.

### Comprueba tu comprensión

**Pregunta:** ¿Qué decisión incorrecta podría producirse si tratamos una dependencia dirigida como una arista no dirigida?

**Responde esta pregunta en notebook.md.**


## 5. Clasificar el dominio de pesos

No basta preguntar “¿hay pesos?”. El dominio completo decide qué invariante es válido:

| Dominio | Ejemplo | Candidato estudiado para caminos |
| --- | --- | --- |
| Sin pesos | cada pasillo cuenta 1 | BFS |
| Solo 0 y 1 | portal gratis o movimiento pagado | 0-1 BFS |
| No negativos generales | tiempos 2, 7, 3.5 | Dijkstra |
| Incluye negativos | bonificación −4 | ninguno de los estudiados |

BFS minimiza número de aristas. 0-1 BFS explota que solo existen dos prioridades posibles. Dijkstra necesita que una distancia extraída como mínima no pueda mejorar mediante una arista negativa.

> [!IMPORTANT]
> Decir “ninguno de los estudiados” es una decisión técnica correcta cuando se viola un contrato. No inventes una solución por presión de entregar un nombre.

Ya tenemos objetivo, dirección y pesos; podemos construir la matriz de decisión.

### Comprueba tu comprensión

**Pregunta:** ¿Por qué el dato 'hay pesos' es insuficiente para elegir entre 0-1 BFS y Dijkstra?

**Responde esta pregunta en notebook.md.**


## 6. Matriz de decisión integradora

La matriz resume contratos; no sustituye el razonamiento previo.

| Objetivo y restricciones | Operación dominante | Estructura | Algoritmo | Costo |
| --- | --- | --- | --- | --- |
| Camino sin pesos | procesar por capas | cola | BFS | O(V+E) |
| Camino con pesos 0/1 | priorizar frente/fondo | deque | 0-1 BFS | O(V+E) |
| Camino con pesos no negativos | extraer menor distancia | heap | Dijkstra | O((V+E) log V) |
| Conexión global mínima, no dirigida | unir componentes con arista barata | Union-Find | Kruskal | O(E log E) |
| Dependencias dirigidas | liberar grado cero | cola + grados | Kahn | O(V+E) |

Observa que la misma complejidad asintótica puede esconder invariantes diferentes. También que compartir estructura no vuelve equivalentes a dos algoritmos.

Antes de avanzar el visualizador, activa **Modo reto** y predice objetivo, estructura y algoritmo. Después revela la decisión y busca el primer supuesto que la justifica.

Ahora aplicaremos la matriz primero a la familia de caminos mínimos.

### Comprueba tu comprensión

**Pregunta:** ¿Qué columna de la matriz explica mejor por qué se eligió una estructura de datos concreta?

**Responde esta pregunta en notebook.md.**


## 7. Laboratorio de decisión interactivo

El panel avanza por once decisiones. La franja superior muestra qué pasos ya tienen evidencia; las dos columnas separan el modelo del problema y la selección técnica.

**Protocolo de uso:**

1. selecciona un caso y activa **Modo reto**;
2. lee el problema sin avanzar;
3. escribe una predicción en `notebook.md`;
4. usa **Siguiente** para revelar una decisión a la vez;
5. compara tu predicción con contrato, prueba distintiva y complejidad;
6. usa **Reproducir/Pausar** solo después de hacer la traza manual.

> [!TIP]
> Si dudas entre dos algoritmos, diseña una entrada donde deberían producir objetivos diferentes. Una prueba distintiva enseña más que una definición memorizada.

Después del panel resolveremos manualmente sus familias de casos.

### Comprueba tu comprensión

**Pregunta:** ¿Qué debes predecir antes de revelar el algoritmo y qué evidencia usarás para corregir tu predicción?

**Responde esta pregunta en notebook.md.**


In [ ]:
from pathlib import Path
import sys
bases=[Path.cwd(),*Path.cwd().parents]
candidatas=[]
for base in bases:
    candidatas.extend([base,base/'clase_20',base/'curso-alumnos'/'clase_20'])
RAIZ_CLASE=next((r for r in candidatas if (r/'src'/'visualizador_decisiones.py').exists()),None)
if RAIZ_CLASE is None:
    raise FileNotFoundError('No se encontró curso-alumnos/clase_20')
sys.path.insert(0,str(RAIZ_CLASE))
from src.visualizador_decisiones import diagnosticar_widgets, mostrar_laboratorio_decisiones
print(diagnosticar_widgets())
mostrar_laboratorio_decisiones()


## 8. Caso de camino sin pesos: BFS

En un edificio cada pasillo cuenta como un paso. Para llegar de `A` a `D`, BFS procesa primero distancia 0, luego distancia 1, después distancia 2.

```text
A: B, C     B: D     C: E     D: —     E: D
capas: {A} → {B,C} → {D,E}
```

Cuando `D` se descubre desde `B`, ninguna ruta con menos aristas puede aparecer después: todas las capas anteriores ya fueron procesadas. La cola conserva justamente ese orden. Un heap podría simular prioridades 0,1,2, pero añadiría costo y complejidad sin necesidad.

**Prueba distintiva.** Construye dos rutas: una con dos aristas largas y otra con tres aristas cortas. Si no hay pesos, “largas” y “cortas” no forman parte del modelo; BFS elige dos aristas.

Ahora permitiremos costos, pero solo los valores 0 y 1.

### Comprueba tu comprensión

**Pregunta:** ¿Qué invariante permite afirmar que el primer descubrimiento de un nodo en BFS usa el mínimo número de aristas?

**Responde esta pregunta en notebook.md.**


## 9. Caso de pesos 0/1: 0-1 BFS

Un portal gratuito puede llevar a un nodo que debe atenderse antes que otro alcanzado pagando 1. El deque implementa dos decisiones:

```text
mejora con peso 0 → agregar al frente
mejora con peso 1 → agregar al final
```

Ejemplo: `A→B(1)`, `A→C(0)`, `C→D(0)`, `B→D(0)`. Tras explorar A, el deque queda `[C, B]`; C se procesa primero y obtiene costo 0 para D. Una cola FIFO habría dejado B delante y ya no representaría prioridad por costo.

0-1 BFS no es “BFS con una condición pequeña”: cambia la estructura porque cambia la operación dominante. Si aparece peso 2, la regla frente/fondo deja de representar todas las prioridades.

El siguiente caso necesita prioridades generales no negativas.

### Comprueba tu comprensión

**Pregunta:** ¿Por qué una mejora de peso 0 entra al frente del deque y una de peso 1 al final?

**Responde esta pregunta en notebook.md.**


## 10. Caso de pesos generales: Dijkstra

Con tiempos 2, 7 o 11 ya no bastan dos extremos. El heap mantiene muchas distancias tentativas y permite retirar la menor.

```text
A→B(8), A→C(2), C→B(1), B→D(3), C→D(9)
```

Aunque B aparece como vecino directo de A, la mejor distancia a B es 3 mediante C. Dijkstra inserta candidatos y descarta entradas obsoletas. Con pesos no negativos, al extraer una distancia mínima vigente no existe una ruta futura que la reduzca pasando por algo aún más costoso.

**Caso distintivo frente a BFS.** La ruta A→B tiene una arista de costo 8; A→C→B tiene dos aristas de costo 3. BFS preferiría una arista; Dijkstra minimiza la suma.

Ya conocemos tres algoritmos de caminos; comparemos sus contratos sin reducirlos a una escala de “mejor”.

### Comprueba tu comprensión

**Pregunta:** ¿Qué operación repetida hace que un heap sea adecuado para Dijkstra?

**Responde esta pregunta en notebook.md.**


## 11. BFS, 0-1 BFS y Dijkstra no forman una competencia

Los tres resuelven caminos mínimos, pero sobre dominios diferentes. La elección más especializada que satisface el contrato suele expresar mejor la información disponible.

| Caso | BFS | 0-1 BFS | Dijkstra |
| --- | --- | --- | --- |
| Sin pesos | apropiado | modelado artificial | correcto con costo extra |
| 0/1 | incorrecto si ignora pesos | apropiado | correcto, menos especializado |
| no negativos generales | incorrecto | contrato inválido | apropiado |
| negativos | incorrecto | inválido | inválido |

“Correcto pero innecesariamente costoso” no es lo mismo que “incorrecto”. Dijkstra puede resolver 0/1, pero 0-1 BFS aprovecha el dominio para O(V+E).

La última fila merece atención: una buena decisión también sabe detenerse.

### Comprueba tu comprensión

**Pregunta:** ¿En qué sentido Dijkstra puede ser correcto pero no la elección más adecuada para pesos 0/1?

**Responde esta pregunta en notebook.md.**


## 12. Pesos negativos: rechazar con precisión

Supón que extraemos `B` con distancia 4 y después una ruta por `C` usa una arista −7 para llegar a B con distancia 1. El argumento central de Dijkstra se rompió: “extraído” ya no significa “definitivo”.

La respuesta responsable contiene tres partes:

1. **contrato violado:** existen pesos negativos;
2. **invariante perdido:** una distancia extraída podría mejorar después;
3. **alcance:** ninguno de los algoritmos estudiados en estas clases resuelve el caso general.

> [!CAUTION]
> No cambies silenciosamente pesos negativos a cero, no uses valor absoluto y no menciones un algoritmo nuevo como si se hubiera estudiado.

Dejamos la familia de caminos y pasamos a un objetivo global: conectar todos los nodos.

### Comprueba tu comprensión

**Pregunta:** ¿Cómo justificarías técnicamente la respuesta 'ninguno de los estudiados' ante pesos negativos?

**Responde esta pregunta en notebook.md.**


## 13. Conexión global: Kruskal y Union-Find

Una empresa necesita conectar todas sus sedes con fibra al menor costo total. No pregunta cómo viajar desde A, sino qué conjunto de enlaces construir.

Kruskal ordena aristas por peso y considera la siguiente más barata. La operación crítica no es encontrar rutas: es saber si los extremos ya pertenecen a la misma componente. Union-Find responde `find(u) == find(v)` y fusiona componentes eficientemente.

```text
aristas: AB(1), CD(1), BC(2), AC(5), AD(8)
aceptar AB → aceptar CD → aceptar BC → detener con V−1 aristas
```

Rechazar una arista que forma ciclo conserva la posibilidad de un árbol. Los pesos negativos no rompen a Kruskal: una arista negativa simplemente aparece antes en el orden.

Ahora contrastaremos este objetivo con Dijkstra, la confusión más importante del cierre.

### Comprueba tu comprensión

**Pregunta:** ¿Qué consulta repetida de Kruskal justifica usar Union-Find?

**Responde esta pregunta en notebook.md.**


## 14. Árbol de caminos mínimos no es MST

Dijkstra puede guardar un predecesor por nodo y formar un árbol desde el origen. Eso no lo convierte en MST. Sus objetivos difieren:

- Dijkstra minimiza cada distancia **desde un origen**;
- Kruskal minimiza la suma de una **red que conecta todos**.

Imagina un centro A conectado a B, C y D por costo 4, mientras B–C y C–D cuestan 1. El árbol de rutas directas desde A cuesta 12. Un MST puede usar A–B, B–C y C–D con costo 6, aunque la ruta A→D ya no sea la mínima desde A.

**Prueba distintiva.** Una entrada útil obliga a los objetivos a discrepar. Si ambos algoritmos producen lo mismo en un triángulo sencillo, ese caso no prueba que sean equivalentes.

La última familia no minimiza pesos: organiza precedencias.

### Comprueba tu comprensión

**Pregunta:** ¿Por qué un árbol de predecesores producido por Dijkstra no tiene que minimizar el costo total de todas sus aristas?

**Responde esta pregunta en notebook.md.**


## 15. Dependencias: Kahn y grados de entrada

Para ejecutar tareas, una tarea está disponible cuando no tiene requisitos pendientes. El grado de entrada cuenta esos requisitos. Kahn encola inicialmente todos los grados cero y, al procesar uno, reduce el grado de sus sucesores.

```text
analizar → implementar → probar
documentar ───────────→ entregar
probar ───────────────→ entregar
```

`entregar` no se libera al terminar solo una rama: entra a la cola exactamente cuando su grado llega a cero. Si al final se procesaron menos de V nodos, quedó un ciclo o una parte bloqueada.

El resultado puede no ser único. Lo correcto es validar que cada arista `u→v` cumple `posición[u] < posición[v]`, no comparar con una lista memorizada.

Kahn comparte cola con BFS, pero el contenido de esa cola significa otra cosa.

### Comprueba tu comprensión

**Pregunta:** ¿Qué significa que un nodo tenga grado de entrada cero durante Kahn?

**Responde esta pregunta en notebook.md.**


## 16. BFS y Kahn comparten cola, no invariante

En BFS, la cola contiene nodos descubiertos pendientes de explorar y conserva capas de distancia. En Kahn, contiene nodos actualmente disponibles y conserva la regla “grado de entrada cero”.

| Pregunta | BFS | Kahn |
| --- | --- | --- |
| ¿Qué entra? | nodo descubierto | nodo con grado cero |
| Estado auxiliar | visitados/distancia | grados de entrada |
| Objetivo | recorrido o camino por aristas | orden de precedencias |
| Señal de problema | destino inalcanzable | cobertura menor que V indica ciclo |

Este contraste demuestra por qué memorizar `cola → BFS` es insuficiente. La estructura soporta operaciones; el algoritmo define estado, invariante y objetivo.

Con las cinco parejas claras, formalizaremos contratos reutilizables.

### Comprueba tu comprensión

**Pregunta:** ¿Qué información adicional a la cola permite distinguir una ejecución de BFS de una de Kahn?

**Responde esta pregunta en notebook.md.**


## 17. Contratos antes de ejecutar

Un contrato separa lo que el algoritmo promete de lo que el llamador debe garantizar.

| Algoritmo | Precondición clave | Poscondición observable |
| --- | --- | --- |
| BFS | aristas con costo uniforme/no modelado | mínimo número de aristas |
| 0-1 BFS | pesos en {0,1} | mínimo costo 0/1 |
| Dijkstra | pesos no negativos | distancias mínimas desde origen |
| Kruskal | grafo ponderado no dirigido | MST o señal de desconexión |
| Kahn | precedencias dirigidas | orden válido o señal de ciclo |

Antes de llamar una implementación existente, valida representación, nodos, pesos y dirección. Después interpreta retornos especiales como `[]`, `None` o infinito según el contrato real de esa clase.

> [!IMPORTANT]
> Reutilizar no significa llamar a ciegas: significa adaptar datos sin cambiar significado y verificar precondiciones.

Ahora veremos cómo reutilizar sin copiar las implementaciones.

### Comprueba tu comprensión

**Pregunta:** ¿Qué responsabilidades conserva el código integrador aunque reutilice una implementación ya probada?

**Responde esta pregunta en notebook.md.**


## 18. Reutilización en lugar de recopia

La Clase 20 no recompensa escribir otra vez los cinco algoritmos. El flujo preferido es:

```python
perfil = describir_problema(enunciado)
decision = seleccionar_estrategia(perfil)
validar_contrato(datos, decision)
resultado = llamar_implementacion_existente(datos)
interpretar(resultado)
```

En el repositorio del profesor hay una demostración que importa directamente las soluciones de las Clases 16–19 y ejecuta un caso pequeño de cada una. Esto permite comprobar integración sin crear copias divergentes.

Si una representación usa nombres y otra índices, escribe un adaptador estrecho y pruébalo. Documenta qué conserva: dirección, pesos, nodos aislados y significado de cada retorno.

La reutilización necesita pruebas que distingan alternativas, no solo casos felices.

### Comprueba tu comprensión

**Pregunta:** ¿Por qué copiar una implementación previa es peor que importarla y adaptar únicamente la entrada y salida?

**Responde esta pregunta en notebook.md.**


## 19. Diseñar pruebas que distinguen decisiones

Una prueba distintiva debe fallar para una alternativa plausible:

- BFS vs Dijkstra: más aristas pero menor costo;
- 0-1 BFS vs BFS: una mejora de peso 0 debe adelantarse;
- Dijkstra vs Kruskal: árbol de caminos desde un origen distinto del MST;
- BFS vs Kahn: un nodo con dos requisitos no se libera tras uno;
- Dijkstra con negativos: una distancia extraída mejora después.

Para cada prueba registra: supuesto, entrada mínima, salida esperada, alternativa descartada y por qué. Agrega también vacío, nodo aislado, desconexión, ciclo y empate cuando correspondan.

> [!TIP]
> Si una prueba solo confirma que el caso fácil funciona para todos los candidatos, no protege la decisión.

Enseguida analizaremos propuestas incorrectas con el mismo formato.

### Comprueba tu comprensión

**Pregunta:** ¿Qué hace que una prueba sea distintiva y no solo un caso de ejemplo?

**Responde esta pregunta en notebook.md.**


## 20. Clínica de selecciones incorrectas

Evalúa estas propuestas sin ejecutarlas:

1. “Usaré BFS porque hay que llegar de A a B”, pero las calles tienen tiempos 2 y 9.
2. “Usaré Dijkstra para conectar todas las sedes”, pero se minimiza el costo total de la red.
3. “Usaré Kahn porque hay una cola”, pero se pide la ruta con menos saltos.
4. “Usaré Kruskal para prerrequisitos”, pero las flechas no son conexiones simétricas.
5. “Usaré Dijkstra aunque existe una arista −3”.

Para corregir cada una usa cuatro frases: objetivo real, contrato violado, operación dominante y elección adecuada —o “ninguno estudiado”. No basta reemplazar un nombre por otro.

Esta clínica prepara el trabajo en equipos: defender una decisión bajo preguntas.

### Comprueba tu comprensión

**Pregunta:** Elige dos propuestas incorrectas y explica objetivo, contrato violado, operación dominante y corrección.

**Responde esta pregunta en notebook.md.**


## 21. Trabajo en equipo A: movilidad urbana

Una aplicación recibe tres versiones de una ciudad:

- **A1:** todas las calles cuentan como un tramo;
- **A2:** cambiar de línea cuesta 1 y continuar cuesta 0;
- **A3:** cada calle tiene minutos no negativos.

El equipo debe producir una tabla con objetivo, dirección, dominio de pesos, operación dominante, estructura, algoritmo, contrato, prueba distintiva y complejidad para cada versión. No programes los algoritmos; diseña adaptadores y tests sobre implementaciones previas.

**Extensión opcional para equipos rápidos.** Explica qué información adicional guardarías para reconstruir la ruta y cómo interpretarías un destino inalcanzable conforme al contrato reutilizado.

Después resolveremos un problema cuyo objetivo no es una ruta.

### Comprueba tu comprensión

**Pregunta:** ¿Cómo cambian estructura y algoritmo entre A1, A2 y A3 aunque el objetivo general siga siendo llegar con costo mínimo?

**Responde esta pregunta en notebook.md.**


## 22. Trabajo en equipo B: construir y planificar

Una universidad plantea dos necesidades:

- tender enlaces entre edificios para que todos queden conectados al menor costo total;
- programar la renovación de cada edificio respetando prerrequisitos técnicos.

Ambos problemas contienen los mismos edificios, pero las aristas significan cosas distintas. Para el primero, propón un caso donde una arista barata forme ciclo y deba rechazarse. Para el segundo, propón un ciclo de dependencias y explica la señal de Kahn.

El equipo presenta durante tres minutos: modelo, elección, una alternativa descartada, test y complejidad. Cada integrante debe poder responder qué cambiaría si la dirección o el objetivo cambia.

Ahora convertiremos la decisión técnica en una explicación evaluable.

### Comprueba tu comprensión

**Pregunta:** ¿Por qué reutilizar los mismos nodos no permite reutilizar automáticamente el mismo algoritmo en las dos necesidades?

**Responde esta pregunta en notebook.md.**


## 23. Comunicación técnica de una decisión

Una explicación defendible puede seguir esta plantilla:

> El problema pide **[objetivo]** sobre un grafo **[dirección/pesos]**. La operación dominante es **[operación]**; por eso usamos **[estructura]** con **[algoritmo]**. Su contrato exige **[precondición]**, que verificamos con **[prueba]**. El costo es **[complejidad]** y la salida significa **[interpretación]**. Descartamos **[alternativa]** porque **[razón]**.

Evita “es más rápido” sin variables, “siempre se usa” sin contrato y explicaciones que solo repiten el nombre del algoritmo. Usa V y E, pero traduce también qué representan en el problema.

> [!NOTE]
> Una buena explicación permite auditar la decisión antes de leer el código.

Con el método completo, haremos una reflexión sobre cómo cambió nuestra forma de resolver problemas.

### Comprueba tu comprensión

**Pregunta:** ¿Qué elementos mínimos debe contener una justificación técnica para que otra persona pueda auditar la elección?

**Responde esta pregunta en notebook.md.**


## 24. Reflexión final del curso

Al inicio del curso, una estructura podía parecer un contenedor. Ahora podemos verla como una decisión sobre operaciones: una cola expresa orden temporal; un heap expresa prioridad; Union-Find expresa componentes; los grados expresan requisitos pendientes.

Completa tres frases con evidencia:

1. “Antes elegía una estructura por ___; ahora primero identifico ___.”
2. “El contrato que más me ayudó a detectar un error fue ___ porque ___.”
3. “Ante un algoritmo que no aplica, ahora puedo ___.”

Revisa tu primera respuesta de este notebook. Corrígela si nombró estructuras sin operaciones o algoritmos sin restricciones. La corrección razonada forma parte del aprendizaje.

La última sección condensará el recorrido en una tabla reutilizable.

### Comprueba tu comprensión

**Pregunta:** ¿Qué cambió en tu proceso de decisión desde la primera clase hasta este laboratorio final?

**Responde esta pregunta en notebook.md.**


## 25. Síntesis y cierre

| Pregunta | Respuesta integradora |
| --- | --- |
| ¿Qué queremos producir? | camino, conexión global u orden |
| ¿Qué describe el modelo? | dirección, pesos y significado de aristas |
| ¿Qué analizamos después? | operación dominante |
| ¿Qué elegimos con ella? | estructura de datos y algoritmo compatibles |
| ¿Qué evita decisiones inválidas? | contrato y precondiciones |
| ¿Qué aporta evidencia? | prueba distintiva |
| ¿Cómo comunicamos costo? | complejidad en V y E, ligada a operaciones |
| ¿Qué hacemos fuera del alcance? | rechazar con precisión, sin inventar |

La ruta completa no termina al obtener una salida: termina cuando podemos explicar qué significa, qué garantiza y bajo qué supuestos es válida.

> [!IMPORTANT]
> **Los algoritmos se vuelven eficientes cuando la estructura de datos coincide con la operación dominante del problema.**

### Comprueba tu comprensión

**Pregunta:** Ante un problema nuevo, ¿cómo identificamos la operación dominante y elegimos la estructura de datos y el algoritmo adecuados?

**Responde esta pregunta en notebook.md.**
